# 📈 Amazon Sales — Data Visualization Notebook
**Libraries:** Matplotlib · Seaborn · Plotly  
**Goal:** Tell the business story with histograms, boxplots, bars, pies, heatmaps, scatter, trends, and interactive charts.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px, plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings; warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

df = pd.read_csv('../Dataset/Cleaned_Data.csv', parse_dates=['order_date'])
print('Rows:', len(df))

## 1. Histograms — Distribution of key metrics

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(16,4))
sns.histplot(df['total_revenue'], bins=40, kde=True, ax=ax[0], color='#FF9900'); ax[0].set_title('Total Revenue')
sns.histplot(df['price'], bins=40, kde=True, ax=ax[1], color='#146EB4'); ax[1].set_title('Price')
sns.histplot(df['rating'], bins=20, kde=True, ax=ax[2], color='#232F3E'); ax[2].set_title('Rating')
plt.tight_layout(); plt.show()

## 2. Boxplots — Outliers & spread across categories

In [ ]:
plt.figure(figsize=(13,5))
sns.boxplot(data=df, x='product_category', y='total_revenue', palette='Set2')
plt.title('Revenue Distribution by Category'); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## 3. Bar Charts — Category & Regional performance

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15,5))
df.groupby('product_category')['total_revenue'].sum().sort_values().plot.barh(ax=ax[0], color='#FF9900')
ax[0].set_title('Revenue by Category'); ax[0].set_xlabel('$')
df.groupby('customer_region')['total_revenue'].sum().sort_values().plot.barh(ax=ax[1], color='#146EB4')
ax[1].set_title('Revenue by Region'); ax[1].set_xlabel('$')
plt.tight_layout(); plt.show()

## 4. Pie Chart — Payment method share

In [ ]:
pay = df['payment_method'].value_counts()
plt.figure(figsize=(7,7))
plt.pie(pay, labels=pay.index, autopct='%1.1f%%', startangle=90, colors=sns.color_palette('Set2'))
plt.title('Order Share by Payment Method'); plt.show()

## 5. Heatmap — Correlation matrix

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df[['price','discount_percent','quantity_sold','rating','review_count',
                'total_revenue','profit','cost']].corr(),
            annot=True, fmt='.2f', cmap='YlGnBu', square=True, linewidths=.5)
plt.title('Feature Correlation Heatmap'); plt.tight_layout(); plt.show()

## 6. Scatter Plots — Price vs Revenue (colored by quantity)

In [ ]:
sample = df.sample(5000, random_state=42)
plt.figure(figsize=(10,6))
sns.scatterplot(data=sample, x='price', y='total_revenue', hue='quantity_sold',
                palette='viridis', alpha=0.6)
plt.title('Price vs Total Revenue (sample 5k)'); plt.tight_layout(); plt.show()

## 7. Trend Charts — Monthly revenue & profit (interactive Plotly)

In [ ]:
trend = df.groupby('year_month').agg(Revenue=('total_revenue','sum'),
                                   Profit=('profit','sum')).reset_index()
fig = make_subplots()
fig.add_trace(go.Scatter(x=trend['year_month'], y=trend['Revenue'], name='Revenue',
                         line=dict(color='#FF9900', width=2)))
fig.add_trace(go.Scatter(x=trend['year_month'], y=trend['Profit'], name='Profit',
                         line=dict(color='#146EB4', width=2)))
fig.update_layout(title='Monthly Revenue & Profit Trend', xaxis_title='Month',
                  yaxis_title='$', hovermode='x unified', template='plotly_white')
fig.show()

## 8. Interactive Charts — Category treemap & Region sunburst

In [ ]:
cat = df.groupby('product_category')['total_revenue'].sum().reset_index()
fig = px.treemap(cat, path=['product_category'], values='total_revenue',
                 color='total_revenue', color_continuous_scale='Blues',
                 title='Revenue Treemap by Category')
fig.update_layout(template='plotly_white'); fig.show()

In [ ]:
reg = df.groupby('customer_region')['total_revenue'].sum().reset_index()
fig = px.bar(reg, x='customer_region', y='total_revenue', color='total_revenue',
             color_continuous_scale='Sunsetdark', title='Revenue by Region (Interactive)')
fig.update_layout(template='plotly_white', xaxis_title='', yaxis_title='$'); fig.show()

### 📣 Business Interpretations
- **Concentration is healthy & diversified** — no single category dominates (>16% each).
- **Beauty = premium-margin engine**; prioritise inventory & cross-sell there.
- **Discounts are dilutive** — visualised margins shrink as discount band rises; switch spend to value-adds.
- **Payment mix is even**, so no single gateway risk; Wallet/UPI lead slightly.
- **Ratings cluster at 3.0** — visual confirms a quality-reputation gap to close.